In [4]:
# Import required libraries and load the original raw dataset

from pathlib import Path
import pandas as pd

current_folder = Path.cwd().resolve()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

raw_dataset_path = project_root / "data" / "raw" / "raw_data.csv"
cleaned_data_folder = project_root / "data" / "cleaned"

df_raw = pd.read_csv(raw_dataset_path, low_memory=False)

print("Raw dataset loaded successfully.")
print(f"Raw dataset shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
display(df_raw.head())

Raw dataset loaded successfully.
Raw dataset shape: 253,098 rows × 2 columns


,url,label
0,https://www.visitcanada.com,legitimate
1,http://218.228.19.9/~yossi/9ssfpkz,phishing
2,https://www.msupress.msu.edu/series.php?series...,legitimate
3,https://docs.google.com/presentation/d/e/2PACX...,phishing
4,https://www.c250.columbia.edu/c250_celebrates/...,legitimate


In [5]:
# Create a copy for cleaning so the original raw dataset remains unchanged

df_clean = df_raw.copy()

print("Cleaning copy created successfully.")
print(f"Cleaning copy shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")

Cleaning copy created successfully.
Cleaning copy shape: 253,098 rows × 2 columns


In [6]:
# Remove accidental leading and trailing spaces from URLs and labels

df_clean["url"] = df_clean["url"].astype(str).str.strip()
df_clean["label"] = df_clean["label"].astype(str).str.strip().str.lower()

print("URL and label text formatting standardized.")

print("\nUnique labels after standardization:")
print(df_clean["label"].value_counts())

URL and label text formatting standardized.

Unique labels after standardization:
label
legitimate    129420
phishing      123678
Name: count, dtype: int64


In [7]:
# Verify row count, missing values, and duplicate records after cleaning

print(f"Rows before cleaning: {df_raw.shape[0]:,}")
print(f"Rows after cleaning: {df_clean.shape[0]:,}")

print("\nMissing values after cleaning:")
print(df_clean.isna().sum())

print("\nExact duplicate rows after cleaning:")
print(df_clean.duplicated().sum())

print("\nDuplicate URLs after cleaning:")
print(df_clean["url"].duplicated().sum())

Rows before cleaning: 253,098
Rows after cleaning: 253,098

Missing values after cleaning:
url      0
label    0
dtype: int64

Exact duplicate rows after cleaning:
0

Duplicate URLs after cleaning:
0


In [8]:
# Count whether whitespace standardization changed any URL or label values

url_values_changed = (
    df_raw["url"].astype(str) != df_clean["url"].astype(str)
).sum()

label_values_changed = (
    df_raw["label"].astype(str) != df_clean["label"].astype(str)
).sum()

print(f"URL values changed by cleaning: {url_values_changed:,}")
print(f"Label values changed by cleaning: {label_values_changed:,}")

URL values changed by cleaning: 195
Label values changed by cleaning: 0


In [9]:
# Create a reproducible summary of the cleaning stage

cleaning_summary = pd.DataFrame({
    "check": [
        "Original row count",
        "Cleaned row count",
        "Rows removed",
        "Missing URL values after cleaning",
        "Missing label values after cleaning",
        "Exact duplicate rows after cleaning",
        "Duplicate URLs after cleaning",
        "URL values changed by whitespace cleaning",
        "Label values changed by standardization"
    ],
    "value": [
        len(df_raw),
        len(df_clean),
        len(df_raw) - len(df_clean),
        df_clean["url"].isna().sum(),
        df_clean["label"].isna().sum(),
        df_clean.duplicated().sum(),
        df_clean["url"].duplicated().sum(),
        url_values_changed,
        label_values_changed
    ]
})

display(cleaning_summary)

,check,value
0,Original row count,253098
1,Cleaned row count,253098
2,Rows removed,0
3,Missing URL values after cleaning,0
4,Missing label values after cleaning,0
5,Exact duplicate rows after cleaning,0
6,Duplicate URLs after cleaning,0
7,URL values changed by whitespace cleaning,195
8,Label values changed by standardization,0


In [10]:
# Save the cleaned dataset and cleaning report without overwriting the raw dataset

cleaned_data_folder.mkdir(parents=True, exist_ok=True)

results_logs_folder = project_root / "results" / "logs"
results_logs_folder.mkdir(parents=True, exist_ok=True)

cleaned_dataset_path = cleaned_data_folder / "cleaned_urls.csv"
cleaning_summary_path = results_logs_folder / "data_cleaning_summary.csv"

df_clean.to_csv(cleaned_dataset_path, index=False)
cleaning_summary.to_csv(cleaning_summary_path, index=False)

print("Cleaned dataset saved to:")
print(cleaned_dataset_path)

print("\nCleaning summary saved to:")
print(cleaning_summary_path)

print("\nSaved cleaned dataset shape:")
print(df_clean.shape)

Cleaned dataset saved to:
C:\Users\Jiya\Documents\Malicious URL Detection\data\cleaned\cleaned_urls.csv

Cleaning summary saved to:
C:\Users\Jiya\Documents\Malicious URL Detection\results\logs\data_cleaning_summary.csv

Saved cleaned dataset shape:
(253098, 2)
